# Machine Learning Midterm - Breast Cancer Wisconsin

This notebook covers Sections 1-10 for the Breast Cancer Wisconsin dataset. Shared functions are in `utils.py`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utils

pd.set_option("display.max_columns", 50)
utils.make_folders()

DATASET = "breast_cancer"
display(utils.package_versions())

,package,version
0,python,3.14.3
1,numpy,2.5.4
2,pandas,3.1.1
3,scikit-learn,1.9.1
4,scipy,1.18.1
5,matplotlib,3.11.2
6,seaborn,0.13.2


# 1. Loading the Dataset
## 1.1 Load the dataset from scikit-learn

In [2]:
X, y, class_names = utils.load_data(DATASET)
print("Loaded with load_breast_cancer()")

Loaded with load_breast_cancer()


### 1.2 Create the data frame (X and y are kept separate)

In [3]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Is the target inside X?", "target" in X.columns)
print("\nFeature names:", list(X.columns))
print("\nTarget labels:", sorted(int(v) for v in y.unique()))
print("Target names:", class_names)
display(X.head())

Shape of X: (569, 30)
Shape of y: (569,)
Is the target inside X? False

Feature names: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']

Target labels: [0, 1]
Target names: ['malignant', 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:
display(utils.class_table(y, class_names))

,label,class_name,n_samples,percent
0,0,malignant,212,37.26
1,1,benign,357,62.74


**Interpretation.** The dataset has 569 samples and 30 numerical features; the target is kept outside X. There are 212 malignant (class 0, 37.3%) and 357 benign (class 1, 62.7%) cases. The classes are moderately imbalanced, so the splits are stratified and macro-averaged metrics are used. Original labels are kept, and malignant (class 0) is the positive class when malignant-case errors are discussed.

## 2. Dataset Quality Checks
### 2.1 Missing values and duplicates

In [5]:
missing = utils.missing_table(X)
display(missing)
print("Total missing values:", int(missing["missing_count"].sum()))
print("Duplicate rows:", utils.count_duplicates(X, y))
print("No missing values and no duplicate rows, so no imputation is needed.")

,feature,missing_count
0,mean radius,0
1,mean texture,0
2,mean perimeter,0
3,mean area,0
4,mean smoothness,0
5,mean compactness,0
6,mean concavity,0
7,mean concave points,0
8,mean symmetry,0
9,mean fractal dimension,0


Total missing values: 0
Duplicate rows: 0
No missing values and no duplicate rows, so no imputation is needed.


### 2.2 Potential outliers (IQR rule and |z| > 3) - nothing is removed

In [6]:
outliers = utils.outlier_table(X)
display(outliers)
print("Features with at least one IQR outlier:", int((outliers["n_iqr_outliers"] > 0).sum()), "of", X.shape[1])

,feature,iqr_lower,iqr_upper,n_iqr_outliers,n_abs_z_gt_3
13,area error,-23.1600,86.2000,65,6
10,radius error,-0.1374,0.8486,38,7
12,perimeter error,-1.0205,5.9835,38,8
23,worst area,-337.7500,1937.0500,35,10
14,smoothness error,0.0007,0.0126,30,7
19,fractal dimension error,-0.0012,0.0080,28,10
15,compactness error,-0.0160,0.0615,28,12
18,symmetry error,0.0027,0.0360,27,11
3,mean area,-123.3000,1326.3000,25,8
29,worst fractal dimension,0.0405,0.1230,24,9


Features with at least one IQR outlier: 29 of 30


### 2.3 Data types

In [7]:
display(utils.dtype_table(X))
n_num = X.select_dtypes(include="number").shape[1]
print("Numerical features:", n_num, "| Categorical features:", X.shape[1] - n_num)

,feature,dtype
0,mean radius,float64
1,mean texture,float64
2,mean perimeter,float64
3,mean area,float64
4,mean smoothness,float64
5,mean compactness,float64
6,mean concavity,float64
7,mean concave points,float64
8,mean symmetry,float64
9,mean fractal dimension,float64


Numerical features: 30 | Categorical features: 0


**Interpretation.** There are no missing values and no duplicate rows. All 30 features are float64 (30 numerical, 0 categorical). 29 of 30 features have at least one IQR outlier; the most are in `area error` (65), `radius error` and `perimeter error` (38 each) and `worst area` (35). These are real large or irregular tumours, so they are kept. KNN and K-Means use Euclidean distance, so very large values such as `worst area` (up to 4254) can dominate the distance. GaussianNB's mean and variance estimates are pulled by extreme values. Decision trees only compare a feature with a threshold, so they are mostly robust to outliers.